Import the required files.

In [2]:
!pip install wordfreq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.8/56.8 MB 18.0 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 18.1 MB/s eta 0:00:00a 0:00:01


In [3]:
from sentence_transformers import SentenceTransformer
import numpy as np
from wordfreq import top_n_list

Normalize the datasize

In [4]:
def normalize(vectors: np.ndarray) -> np.ndarray:
    norms = np.linalg.norm(vectors, axis=1, keepdims=True)
    norms[norms == 0] = 1.0
    return vectors / norms

Single word translator class

This is a multilingual model for sentence embeddings & translation. -->


In [5]:
class SingleWordTranslator:
    def __init__(self, model_name: str = "sentence-transformers/distiluse-base-multilingual-cased-v1"):
        self.model = SentenceTransformer(model_name)

    def embed(self, words):
        embeddings = self.model.encode(words, convert_to_numpy=True, show_progress_bar=False)
        return normalize(embeddings)

    # Creates the source and target embeddings, then calculates the cosine similarity between them.
    # Returns the top k translations for each source word.

    def top_k_translations(self, source_words, target_words, k=5):
        source_embeddings = self.embed(source_words)
        target_embeddings = self.embed(target_words)

        similarities = np.dot(source_embeddings, target_embeddings.T)
        top_k_indices = np.argsort(-similarities, axis=1)[:, :k]

        results = {}
        for i, indices in enumerate(top_k_indices):
            results[source_words[i]] = [(target_words[j], float(similarities[i][j])) for j in indices]

        return results


Establish the word sets to use.

In [6]:
english_words = ["cat", "dog", "house", "car", "tree", "book", "computer", "phone", "city", "river", "mountain", "ocean", "music", "art", "science", "history", "food", "travel", "friend", "family", "love", "happiness", "work", "school", "nature", "health", "money", "time", "life", "dream", "hope"]
german_words = ["Katze", "Hund", "Haus", "Auto", "Baum", "Buch", "Computer", "Telefon", "Stadt", "Fluss", "Berg", "Ozean", "Musik", "Kunst", "Wissenschaft", "Geschichte", "Essen", "Reisen", "Freund", "Familie", "Liebe", "Glück", "Arbeit", "Schule", "Natur", "Gesundheit", "Geld", "Zeit", "Leben", "Traum", "Hoffnung"]

Run translation and print results

In [7]:
translator = SingleWordTranslator()
translations = translator.top_k_translations(english_words, german_words, k=5)

for eng_word, candidates in translations.items():
    print(f"English word: {eng_word}")
    
    print("Top German candidates:")

    for german_word, score in candidates:
        print(f"  German candidate: {german_word}, Similarity score: {score:.4f}")
    

modules.json:   0%|          | 0.00/341 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/556 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/114 [00:00<?, ?B/s]

2_Dense/model.safetensors:   0%|          | 0.00/1.58M [00:00<?, ?B/s]

English word: cat
Top German candidates:
  German candidate: Katze, Similarity score: 0.9657
  German candidate: Hund, Similarity score: 0.6382
  German candidate: Baum, Similarity score: 0.4786
  German candidate: Essen, Similarity score: 0.4401
  German candidate: Liebe, Similarity score: 0.4376
English word: dog
Top German candidates:
  German candidate: Hund, Similarity score: 0.9825
  German candidate: Katze, Similarity score: 0.6152
  German candidate: Freund, Similarity score: 0.4910
  German candidate: Geld, Similarity score: 0.4458
  German candidate: Haus, Similarity score: 0.4421
English word: house
Top German candidates:
  German candidate: Haus, Similarity score: 0.9598
  German candidate: Familie, Similarity score: 0.5642
  German candidate: Schule, Similarity score: 0.4750
  German candidate: Hund, Similarity score: 0.4545
  German candidate: Natur, Similarity score: 0.4479
English word: car
Top German candidates:
  German candidate: Auto, Similarity score: 0.8873
  Germ

Now, try it with your own english or german word!
For this we will import the 50,000 most common english and german words. This will take us more time to run.

In [8]:
english_words = top_n_list("en", 50000) # english
german_words = top_n_list("de", 50000)  # deutsch is de, this means german

Translates words, this is mostly a UI function tbh

Note this translate_word() function was having english words pop up in the german results, this may be because it is just a frequency count and english words are more likely to appear in german texts than vice versa. English is very common. It could also be an unnoticed error, but just reference the other examples seen above and below for proof of concept.

In [9]:
def tranlate_word():
    print("\nNow, try it with your own english or german word!")

    language = input("Would you like to translate English words to German or German words to English? (Type 'E2G' or 'G2E'): ").strip().upper()

    if language == 'E2G':
        source_word = input("Enter an English word to translate to German: ").strip()
        source_words = [source_word]
        target_words = german_words
        target_label = "German"

    elif language == 'G2E':
        source_word = input("Enter a German word to translate to English: ").strip()
        source_words = [source_word]
        target_words = english_words
        target_label = "English"

    else:
        raise ValueError("Invalid input. Please type 'E2G' or 'G2E'.")

    translator = SingleWordTranslator()
    translations = translator.top_k_translations(source_words, target_words, k=5)

    for source_words, candidates in translations.items():

        print(f"Source word: {source_words}")
        
        print(f"The top {target_label} candidates are: ")
        
        for target_word, score in candidates:
            print(f"  {target_label} candidate: {target_word}, Similarity score: {score:.4f}")
            

This is an example of translating "schadenfreude" to English, which doesn't really translate that well. Notice the top similarity score is only 0.8026 (seems high, but relatively low).
Just look at how many words are given within 0.03% of each other: unhappiness, unhappy, demoralizing, demoralized, dissatisfaction are all "kinda" the translation, but not quite.

In [11]:
tranlate_word()


Now, try it with your own english or german word!
Source word: work
The top German candidates are: 
  German candidate: ¦, Similarity score: 0.8685
  German candidate: ツ, Similarity score: 0.8647
  German candidate: work, Similarity score: 0.8617
  German candidate: ä, Similarity score: 0.8497
  German candidate: á, Similarity score: 0.8494


Now, an example of a good translation. Ocean and Ozean are obviously the direct translations and translate well. Note they have a relatively much higher cosine similarity.

In [92]:
tranlate_word()


Now, try it with your own english or german word!
Source word: Ozean
The top English candidates are: 
  English candidate: ocean's, Similarity score: 0.9350
  English candidate: ocean, Similarity score: 0.9302
  English candidate: oceans, Similarity score: 0.9208
  English candidate: oceanside, Similarity score: 0.9194
  English candidate: oceania, Similarity score: 0.9059


References: 

https://sbert.net/

https://sbert.net/docs/sentence_transformer/usage/usage.html

https://huggingface.co/sentence-transformers/distiluse-base-multilingual-cased-v1


Now let's try to compare english sentence to german sentence.

In [12]:
from sentence_transformers import SentenceTransformer
import numpy as np

# Load the same multilingual model
model = SentenceTransformer("sentence-transformers/distiluse-base-multilingual-cased-v1")

def normalize(vectors: np.ndarray) -> np.ndarray:
    norms = np.linalg.norm(vectors, axis=1, keepdims=True)
    norms[norms == 0] = 1.0
    return vectors / norms

def sentence_similarity(sentences1, sentences2):
    # Encode each list
    emb1 = normalize(model.encode(sentences1, convert_to_numpy=True))
    emb2 = normalize(model.encode(sentences2, convert_to_numpy=True))

    # Pairwise cosine similarities
    sims = np.dot(emb1, emb2.T)
    return sims

# Example: English vs German sentence
eng = "Do you speak German?"
ger = "Sprechen Sie Deutsch?"

sim = sentence_similarity([eng], [ger])[0][0]
print(f"Similarity between:\n'{eng}'\n'{ger}'\n→ {sim:.4f}")


Similarity between:
'Do you speak German?'
'Sprechen Sie Deutsch?'
→ 0.7828


In [13]:
eng = "I love music."
german_candidates = [
    "Ich liebe Musik.",
    "Das Wetter ist schön.",
    "Musik macht mich glücklich."
]

sims = sentence_similarity([eng], german_candidates)[0]
for g, s in zip(german_candidates, sims):
    print(f"{g} → {s:.4f}")


Ich liebe Musik. → 0.7339
Das Wetter ist schön. → 0.2617
Musik macht mich glücklich. → 0.5588
